# Modélisation de Data Warehouse

## Objéctifs :
- Création de tables avec clé primaires et clé étrangères, contraintes et comment cela fonctionne dans Databricks
- Insertion des données TPC-H (présentes dans le schéma bronze)
- Visualiser les échecs de chargement de données en cas de non respect des contraintes
- Fournir des étapes permettant le retour en arrière pour des tables
- Visualisation des schéma ER (Entité- Relation) dans Databricks

## Setup
- Créer un catalag **demo_username** si il n'existe pas
- Créer les schéma **bronze, silver, gold** dans le catalog
- importe les tables de **samples.tpch** dans le schéma bronze

In [0]:
%run "./Resources/NB01/Setup"

In [0]:
# récupération du nom du catalog
catalog_name = "demo_" + spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]

# Définition du schéma à utiliser
schema_name = "silver"

# On renseigne les catalogue / schéma à utiliser
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {schema_name}")

display(f"Catalog : {catalog_name}")
display(f"Schema : {schema_name}")


## Création de tables avec contraintes

On va créer 2 tables pour utiliser les clés primaires (PK) et clés étrangères (FK)
1. `lab_customer` avec un clé primaire sur `c_custkey`
2. `lab_orders` avec :
   - un clé primaire sur o_orderkey
   - un clé étrangère sur o_custkey référencant (c_custkey)

> **Note :** Databricks ne fait pas respecter (enforce) les contraintes de clé primaire (PK) et de clé étrangère (FK) au niveau du moteur SQL. Les contraintes sont déclaratives et servent principalement à la documentation et à la génération de schémas, mais elles ne bloquent pas l'insertion de données non conformes.

C'est contraintes sont à titre d'information, cela permet à databricks de fournir des features comme : Diagrammes Entité-Relation (ERD) dans l'explorateur de catalogues, qui affiche la rélation clé primaire et étrangère comme des graphes.

Dans Databricks SQL, les contraintes appliquées ("enforced") au niveau du moteur sont limitées :

- NOT NULL : La contrainte NOT NULL est strictement appliquée – si une colonne est définie NOT NULL, toute tentative d’insertion ou de modification avec une valeur NULL dans cette colonne échouera.

- CHECK (Databricks 16.x+ avec Delta tables) : Depuis les versions récentes, certaines contraintes CHECK peuvent être appliquées et vérifiées lors de l’écriture (par exemple, pour une expression sur une colonne), ce qui provoque une erreur si la condition n’est pas respectée.

- PRIMARY KEY / UNIQUE / FOREIGN KEY : Ces contraintes sont déclaratives dans Databricks : elles sont conservées dans le schéma, mais ne sont pas strictement appliquées lors des insertions ou modifications. Cela veut dire qu’il est possible d’insérer des doublons ou de violer les relations sans erreur.

- DEFAULT : Les valeurs par défaut sont renseignées si aucune valeur n’est fournie lors de l’insertion, mais cela n’enforce pas de validation supplémentaire.




In [0]:
%sql

-- créer la table lab_customer avec un clé primaire sur c_custkey

CREATE TABLE IF NOT EXISTS lab_customer
(
  c_custkey INT,
  c_name STRING,
  c_address STRING,
  c_nationkey INT,
  c_phone STRING,
  c_acctbal DECIMAL(12,2),
  c_mktsegment STRING,
  c_comment STRING,
  CONSTRAINT pk_custkey PRIMARY KEY (c_custkey)
);

In [0]:
%sql 

-- création de la table lab_orders avec un clé primaire sur o_orderkey et une fk sur o_custkey

CREATE TABLE IF NOT EXISTS lab_orders
(
  o_orderkey INT,
  o_custkey INT,
  o_orderstatus STRING,
  o_totalprice DECIMAL(12,2),
  o_orderdate DATE,
  o_orderpriority STRING,
  o_clerk STRING,
  o_shippriority INT,
  o_comment STRING,
  CONSTRAINT pk_orderkey PRIMARY KEY (o_orderkey),
  CONSTRAINT fk_custkey FOREIGN KEY (o_custkey) REFERENCES lab_customer(c_custkey)
);

## Insértion des données de TPC-H depuis les tables bronzes

In [0]:
%sql
-- insertion des données dans lab_customer depuis bronze.tpch_customer
INSERT INTO lab_customer
SELECT
  c_custkey,
  c_name,
  c_address,
  c_nationkey,
  c_phone,
  c_acctbal,
  c_mktsegment,
  c_comment
FROM
  bronze.tpch_customer;

In [0]:
%sql
-- insertion des données dans lab_orders depuis bronze.tpch_orders
INSERT INTO lab_orders
SELECT
  o_orderkey,
  o_custkey,
  o_orderstatus,
  o_totalprice,
  o_orderdate,
  o_orderpriority,
  o_clerk,
  o_shippriority,
  o_comment
FROM
  bronze.tpch_orders;

## Violation des contraintes

Comme databricks ne force pas la vérification des contraintes sur les PK et FK on va vérifier : 
1. **Foreign Key Violation** : insertion d'une ligne dans lab_orders référencant un o_custkey inéxistant.
2. **Primary Key Violation** : insertion d'un doublons dans lab_customer 

### Violation de contrainte de clé étrangère

In [0]:
%sql
INSERT INTO lab_orders
VALUES
(
  999999,   -- o_orderkey
  999999,   -- o_custkey
  'F',      -- o_orderstatus
  1000.00,
  current_date(),
  '3-LOW',
  'Clerk#0000001',
  0,
  'Test de c_custkey inéxistante'
);

### Violation de clé primaire

In [0]:
%sql
INSERT INTO lab_customer
VALUES
(
  1,   -- c_custkey
  'Doublon',   -- c_name
  'Doublon',   -- c_address
  9999,   -- c_nationkey
  '999-999-9999',   -- c_phone
  1000.00,
  'Doublon',
  'Insertion de doublons'
);

In [0]:
%sql
-- retour en arrière
delete from lab_customer where c_name = 'Doublon';
delete from lab_orders where o_orderkey = 999999;

## Visualisation des diagrammes ER dans Databricks

1. Naviguer vers l'onglet **Catalog**
2. Séléctionner le catalog demo_username et schéma (silver)
3. Cliquer sur la table lab_orders et ensuite sur `View relationships`

In [0]:
%sql
DROP TABLE IF EXISTS lab_customer;
DROP TABLE IF EXISTS lab_orders;